# Übung 10 — SQL Schritt für Schritt zum Prüfungsniveau AP01 (Übung)

Dieses Notebook führt dich in **8 Stufen** von einfachen Abfragen bis zum Niveau der Prüfung
*AP01 Management & Nutzung relationaler Daten*:

| Stufe | Thema | Prüfungsteil |
|---|---|---|
| 1 | Datenmodell verstehen (PK, FK, Beziehungen) | A |
| 2 | Eine Tabelle abfragen | C |
| 3 | Aggregieren, Runden, `HAVING` | C |
| 4 | Tabellen verbinden (2 und 3 Tabellen) | C |
| 5 | `LEFT JOIN` und typische Fallen | B |
| 6 | Datenqualität mit SQL prüfen | A, C |
| 7 | Fehleranalyse: Abfragen korrigieren und begründen | B |
| 8 | Konzeptfragen mit Bezug zum Schema beantworten | A |

Danach bist du bereit für die **Probeprüfung** `11_probepruefung_ap01.ipynb`.

**So arbeitest du:** Schreibe deine SQL-Abfrage in die Zelle mit `-- TODO`. Die Zellen **Selbstkontrolle**
prüfen das Ergebnis. Bei Begründungsaufgaben schreibst du deine Antwort in die Markdown-Zelle.

## Das Datenmodell der Fitnesskette «MoveFit»

Fiktive Daten, Details in `Data/fitness_beschreibung.md`. Pfeile zeigen von der Tabelle mit dem Fremdschlüssel
auf die referenzierte Tabelle (n : 1).

```
 studios (studioid PK, name, city, opened_year)
   ^                 ^
   | studioid        | studioid
 trainers            |
 (trainerid PK,      |               courses (courseid PK, title, category,
  firstname,         |                        duration_min, level)
  lastname,          |                  ^
  studioid FK,       |                  | courseid
  hourly_rate)       |                  |
   ^                 |                  |
   | trainerid       |                  |
   +---------- sessions (sessionid PK, courseid FK, studioid FK, trainerid FK,
                         session_date, start_time, capacity, base_price)
                         ^
                         | sessionid
               bookings (bookingid PK, sessionid FK, memberid FK,
                         booking_date, paid_price, status)
                         | memberid
                         v
               members  (memberid PK, firstname, lastname, city, birth_year,
                         membership_type, signup_date, email)
```

- `base_price` = Listenpreis eines Kurstermins, `paid_price` = tatsächlich bezahlter Preis einer Buchung
- gültige `membership_type`: `Basic`, `Premium`, `Student`, `Senior`
- `status` einer Buchung: `attended`, `no-show`, `cancelled` (stornierte Buchungen werden zurückerstattet)

## Setup (wird nicht bewertet)

In [ ]:
# Libraries
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

In [ ]:
# Zugangsdaten (Werte aus docker-compose.yml, über Umgebungsvariablen überschreibbar)
user     = os.getenv('PG_USER', 'pgadmin')
password = os.getenv('PG_PASSWORT', 'geheim')
port     = os.getenv('PG_PORT', '5432')
database = os.getenv('PG_DATABASE', 'postgres')

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in (os.getenv('PG_HOST'), 'db', 'localhost'):
    if not candidate:
        continue
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

if host:
    db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'
else:
    # Backup: SQLite-Datei verwenden (gleiche Tabellen und Inhalte)
    print('Keine PostgreSQL-Verbindung gefunden -> verwende das SQLite-Backup Data/fitness.db')
    db_connection_url = 'sqlite:///Data/fitness.db'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Verbindung:', engine.dialect.name, '| Host:', host)
engine.dispose()

In [ ]:
# Tabellen aus der SQLite-Datei fitness.db nach PostgreSQL übertragen
tables = ['studios', 'courses', 'trainers', 'sessions', 'members', 'bookings']

if db_connection_url.startswith('postgresql'):
    sqlite_engine = create_engine('sqlite:///Data/fitness.db')
    engine = create_engine(db_connection_url)
    for table in tables:
        df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
        df.to_sql(table, engine, if_exists='replace', index=False)
        print(f'{table:<10}{len(df):>6} Zeilen')
    sqlite_engine.dispose()
    engine.dispose()

Die Hilfsfunktion `abfrage()` führt eine SQL-Abfrage aus und gibt das Ergebnis als DataFrame zurück.
In der Prüfung schreibst du dieselben drei Schritte jeweils aus:
`engine = create_engine(...)` → `pd.read_sql_query(..., con=engine)` → `engine.dispose()`.

In [ ]:
def abfrage(sql):
    engine = create_engine(db_connection_url)
    df = pd.read_sql_query(sql, con=engine)
    engine.dispose()
    return df

---
## Stufe 1 — Das Datenmodell verstehen

### 1.1 Wie viele Zeilen hat jede Tabelle?
Zähle mit einer Schleife und `SELECT COUNT(*) AS n FROM ...` die Zeilen jeder Tabelle in `tables` und speichere sie
im Dictionary `anzahl` (Tabellenname → Anzahl).

In [ ]:
anzahl = {}
# TODO: Schleife über tables

anzahl

In [ ]:
# Selbstkontrolle 1.1
assert anzahl == {'studios': 6, 'courses': 15, 'trainers': 30, 'sessions': 300,
                  'members': 505, 'bookings': 2000}
print('1.1 richtig ✔')

### 1.2 Schlüssel und Beziehungen (Konzept)
a) Was ist der Primärschlüssel von `bookings`, welche Fremdschlüssel hat die Tabelle und worauf verweisen sie?

b) In welcher Beziehung stehen `members` und `sessions` zueinander (1:1, 1:n, n:m)? Welche Tabelle löst die
Beziehung auf?

c) Die Spalte `trainers.studioid` ist bei einem Trainer leer (NULL). Ist das ein Verstoss gegen die referenzielle
Integrität?

**Ihre Antwort:**

> [Hier eintragen]

### 1.3 Ein erster Blick
Zeige die ersten 5 Zeilen der Tabelle `sessions` (alle Spalten), sortiert nach `sessionid`. Ergebnis: `df_13`.

In [ ]:
df_13 = abfrage("""
-- TODO
""")
df_13

In [ ]:
# Selbstkontrolle 1.3
assert df_13.shape == (5, 8) and df_13['sessionid'].tolist()[0] == 'SE0001'
print('1.3 richtig ✔')

---
## Stufe 2 — Eine Tabelle abfragen

### 2.1 Alle Yoga-Kurse
Gib `title` und `duration_min` aller Kurse der Kategorie `Yoga` aus, sortiert nach `title`. → `df_21`

In [ ]:
df_21 = abfrage("""
-- TODO
""")
df_21

In [ ]:
# Selbstkontrolle 2.1
assert df_21['title'].tolist() == ['Power Yoga', 'Yin Yoga']
print('2.1 richtig ✔')

### 2.2 Städte der Mitglieder
Aus welchen verschiedenen Städten kommen die Mitglieder? Alphabetisch sortiert, Spalte `city`. → `df_22`

In [ ]:
df_22 = abfrage("""
-- TODO
""")
df_22

In [ ]:
# Selbstkontrolle 2.2
assert df_22['city'].tolist() == ['Basel', 'Bern', 'Biel', 'Luzern', 'Olten', 'Thun', 'Winterthur', 'Zürich']
print('2.2 richtig ✔')

### 2.3 Die teuersten Kurstermine
Gib die 5 Kurstermine mit dem höchsten `base_price` aus (`sessionid`, `courseid`, `studioid`, `base_price`).
Bei gleichem Preis nach `sessionid` aufsteigend sortieren. → `df_23`

In [ ]:
df_23 = abfrage("""
-- TODO
""")
df_23

In [ ]:
# Selbstkontrolle 2.3
assert len(df_23) == 5 and (df_23['base_price'] == 35.0).all()
assert set(df_23['courseid']) == {'CO09'} and set(df_23['studioid']) == {'ST02'}
print('2.3 richtig ✔')

### 2.4 Mehrere Bedingungen
Wie viele Mitglieder mit `membership_type` `Student` wohnen in `Bern` **oder** `Thun`? Gib die Anzahl als Spalte
`n` aus. Achte auf Klammern! → `df_24`

In [ ]:
df_24 = abfrage("""
-- TODO
""")
df_24

### 2.5 Textmuster
Gib `memberid`, `firstname`, `lastname` aller Mitglieder aus, deren Nachname mit `B` beginnt und die **nach 2000**
geboren sind, sortiert nach `lastname`, `firstname`. → `df_25`

In [ ]:
df_25 = abfrage("""
-- TODO
""")
df_25

---
## Stufe 3 — Aggregieren, Runden, HAVING

### 3.1 Buchungen pro Status
Wie viele Buchungen gibt es pro `status`? Spalten `status`, `n`, absteigend nach `n`. → `df_31`

In [ ]:
df_31 = abfrage("""
-- TODO
""")
df_31

In [ ]:
# Selbstkontrolle 3.1
assert dict(zip(df_31['status'], df_31['n'].astype(int))) == {'attended': 1685, 'no-show': 168, 'cancelled': 147}
print('3.1 richtig ✔')

### 3.2 Durchschnittspreis runden — die PostgreSQL-Falle
Berechne den durchschnittlichen `base_price` aller Kurstermine als `avg_price`, gerundet auf 2 Dezimalstellen.

1. Versuche zuerst `ROUND(AVG(base_price), 2)`. Was passiert in PostgreSQL?
2. Löse das Problem mit `ROUND(CAST(AVG(base_price) AS NUMERIC), 2)`. → `df_32`

In [ ]:
df_32 = abfrage("""
-- TODO
""")
df_32

### 3.3 Gruppen filtern mit HAVING
Welche Studios (`studioid`) haben **mehr als 55** Kurstermine? Spalten `studioid`, `session_count`, absteigend
nach `session_count`. → `df_33`

In [ ]:
df_33 = abfrage("""
-- TODO
""")
df_33

### 3.4 Mehrere Kennzahlen pro Gruppe
Berechne pro `category` der Tabelle `courses` die Anzahl Kurse (`n_courses`) und die durchschnittliche Dauer
(`avg_duration`, gerundet auf 1 Stelle). Sortiert nach `n_courses` absteigend, dann `category`. → `df_34`

In [ ]:
df_34 = abfrage("""
-- TODO
""")
df_34

In [ ]:
# Selbstkontrolle 3.4
assert df_34['category'].tolist()[:2] == ['Kraft', 'Gesundheit']
assert dict(zip(df_34['category'], df_34['n_courses'].astype(int)))['Kraft'] == 4
assert float(df_34.set_index('category').loc['Yoga', 'avg_duration']) == 67.5
print('3.4 richtig ✔')

---
## Stufe 4 — Tabellen verbinden

### 4.1 Zwei Tabellen
Gib für alle Kurstermine im Studio `ST03` im **Januar 2026** `sessionid`, `session_date` und den Kurstitel `title`
aus, sortiert nach `session_date`, `sessionid`.

*Tipp:* `session_date` ist als Text `JJJJ-MM-TT` gespeichert — `BETWEEN '2026-01-01' AND '2026-01-31'` funktioniert.
→ `df_41`

In [ ]:
df_41 = abfrage("""
-- TODO
""")
df_41

### 4.2 Drei Tabellen
Wie viele Buchungen hat jedes Studio? `bookings` ist **nicht direkt** mit `studios` verbunden — der Weg führt über
`sessions`. Spalten `name`, `booking_count`, absteigend sortiert. → `df_42`

In [ ]:
df_42 = abfrage("""
-- TODO
""")
df_42

In [ ]:
# Selbstkontrolle 4.2
assert df_42['name'].tolist()[0] == 'MoveFit Oerlikon' and int(df_42['booking_count'][0]) == 560
assert len(df_42) == 5   # MoveFit Wankdorf fehlt -> siehe Stufe 5!
print('4.2 richtig ✔')

### 4.3 Umsatz pro Kurskategorie
Berechne den Umsatz (`SUM(paid_price)`) pro Kurskategorie. **Stornierte Buchungen** (`status = 'cancelled'`) werden
zurückerstattet und zählen nicht. Spalten `category`, `revenue` (2 Stellen), absteigend. → `df_43`

In [ ]:
df_43 = abfrage("""
-- TODO
""")
df_43

### 4.4 Text verketten
Welche 5 Trainer haben am meisten Kurstermine geleitet? Gib `trainerid`, den vollen Namen als `trainer_name`
(`firstname || ' ' || lastname`) und `session_count` aus. Sortierung: `session_count` absteigend, dann `trainerid`.
→ `df_44`

In [ ]:
df_44 = abfrage("""
-- TODO
""")
df_44

In [ ]:
# Selbstkontrolle 4.4
assert df_44['trainerid'].tolist() == ['TR001', 'TR007', 'TR017', 'TR030', 'TR022']
assert df_44['trainer_name'][0] == 'Noah Gerber'
print('4.4 richtig ✔')

*Frage zum Nachdenken:* Warum steht in `GROUP BY` neben `t.trainerid` auch `t.firstname, t.lastname`?

**Antwort:**

> [Hier eintragen]

---
## Stufe 5 — LEFT JOIN und typische Fallen

### 5.1 Alle Studios — auch ohne Buchungen
In 4.2 fehlt das neue Studio *MoveFit Wankdorf*. Schreibe die Abfrage so um, dass **alle 6 Studios** erscheinen und
Wankdorf mit `0` ausgewiesen wird. → `df_51`

In [ ]:
df_51 = abfrage("""
-- TODO
""")
df_51

In [ ]:
# Selbstkontrolle 5.1
assert len(df_51) == 6
assert dict(zip(df_51['name'], df_51['booking_count'].astype(int)))['MoveFit Wankdorf'] == 0
print('5.1 richtig ✔')

### 5.2 COUNT(*) gegen COUNT(spalte)
Führe 5.1 noch einmal mit `COUNT(*)` statt `COUNT(b.bookingid)` aus → `df_52`. Welchen Wert erhält Wankdorf und warum?

In [ ]:
df_52 = abfrage("""
-- TODO
""")
df_52

In [ ]:
# Selbstkontrolle 5.2
assert dict(zip(df_52['name'], df_52['booking_count'].astype(int)))['MoveFit Wankdorf'] == 1
print('5.2 richtig ✔')

**Begründung:**

> [Hier eintragen]

### 5.3 Was fehlt? (Anti-Join)
a) Welche Kurse wurden **noch nie** angeboten? Spalten `courseid`, `title`. → `df_53a`

b) Wie viele Mitglieder haben **noch nie** gebucht? Spalte `n`. → `df_53b`

In [ ]:
df_53a = abfrage("""
-- TODO
""")
df_53b = abfrage("""
-- TODO
""")
display(df_53a, df_53b)

In [ ]:
# Selbstkontrolle 5.3
assert df_53a['title'].tolist() == ['Aqua Fit']
assert int(df_53b['n'][0]) == 20
print('5.3 richtig ✔')

### 5.4 Fehlende Zuordnung
Liste **alle 30** Trainer mit `trainerid` und dem Namen ihres Studios (`studio_name`) auf, sortiert nach `trainerid`.
Ein Trainer hat kein Studio — er muss trotzdem erscheinen. → `df_54`

In [ ]:
df_54 = abfrage("""
-- TODO
""")
df_54.tail()

In [ ]:
# Selbstkontrolle 5.4
assert len(df_54) == 30 and df_54['studio_name'].isnull().sum() == 1
print('5.4 richtig ✔')

---
## Stufe 6 — Datenqualität mit SQL prüfen

Typische Data-Quality-Eigenschaften (Vorlesung): **accurate** (korrekt, entspricht der Realität),
**consistent** (gleiche Information gleich dargestellt, keine Widersprüche), **complete** (vollständig, keine
fehlenden Werte), **reliable** (vertrauenswürdig, nachvollziehbare Herkunft).

### 6.1 Ungültige Kategorien
Gib alle Mitglieder (`memberid`, `firstname`, `lastname`, `membership_type`) mit einem **ungültigen**
`membership_type` aus (gültig: `Basic`, `Premium`, `Student`, `Senior`), sortiert nach `memberid`. → `df_61`

In [ ]:
df_61 = abfrage("""
-- TODO
""")
df_61

In [ ]:
# Selbstkontrolle 6.1
assert df_61['membership_type'].tolist() == ['basic', 'BASIC', 'Basik', 'premium', 'student']
print('6.1 richtig ✔')

### 6.2 Reicht Normalisieren?
Zähle die Mitglieder pro `LOWER(membership_type)` (Spalten `typ`, `n`, sortiert nach `typ`). → `df_62`

Welcher ungültige Wert bleibt trotz `LOWER()` übrig, und welche Data-Quality-Eigenschaft ist verletzt?

In [ ]:
df_62 = abfrage("""
-- TODO
""")
df_62

In [ ]:
# Selbstkontrolle 6.2
assert 'basik' in df_62['typ'].tolist() and len(df_62) == 5
print('6.2 richtig ✔')

**Antwort:**

> [Hier eintragen]

### 6.3 Die NULL-Falle
Gib die Anzahl Mitglieder aus, deren E-Mail **fehlt oder ungültig** ist. Als ungültig gilt: leer, ohne `@` oder ohne
Punkt nach dem `@` (Muster `'%@%.%'`).

1. Probiere zuerst nur `WHERE email NOT LIKE '%@%.%'` → `df_63a`
2. Korrigiere die Abfrage → `df_63b`. Warum unterscheiden sich die Ergebnisse?

In [ ]:
df_63a = abfrage("""
-- TODO
""")
df_63b = abfrage("""
-- TODO
""")
display(df_63a, df_63b)

In [ ]:
# Selbstkontrolle 6.3
assert int(df_63a['n'][0]) == 3 and int(df_63b['n'][0]) == 18
print('6.3 richtig ✔')

### 6.4 Plausibilitätsprüfungen
Schreibe je eine Abfrage, die die Anzahl betroffener Zeilen als `n` liefert:

a) Mitglieder mit unplausiblem Geburtsjahr (vor 1920 oder nach 2026) → `df_64a`

b) Buchungen, bei denen `paid_price` **höher** ist als der `base_price` des Termins → `df_64b`

c) Buchungen, deren `booking_date` **nach** dem `session_date` liegt → `df_64c`

In [ ]:
df_64a = abfrage("""
-- TODO
""")
df_64b = abfrage("""
-- TODO
""")
df_64c = abfrage("""
-- TODO
""")
display(df_64a, df_64b, df_64c)

In [ ]:
# Selbstkontrolle 6.4
assert [int(d['n'][0]) for d in (df_64a, df_64b, df_64c)] == [2, 2, 3]
print('6.4 richtig ✔')

### 6.5 Überbuchte Kurstermine
Welche Kurstermine haben mehr **nicht stornierte** Buchungen als Plätze (`capacity`)? Spalten `sessionid`,
`capacity`, `booking_count`, sortiert nach `sessionid`. → `df_65`

In [ ]:
df_65 = abfrage("""
-- TODO
""")
df_65

In [ ]:
# Selbstkontrolle 6.5
assert df_65['sessionid'].tolist() == ['SE0005', 'SE0007', 'SE0114']
print('6.5 richtig ✔')

---
## Stufe 7 — Fehleranalyse (wie Prüfungsteil B)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Ergebnis.* Schreibe die korrigierte
Abfrage und **begründe jede Änderung**. Prüfungstipp: Nenne immer (1) was die ursprüngliche Abfrage tut,
(2) warum das zum falschen Ergebnis führt, (3) was du änderst.

### 7.1 Trainer pro Studio
**Ziel:** Anzahl Trainer pro Studio — für **alle 6** Studios.
```sql
SELECT st.name, COUNT(*) AS trainer_count
FROM studios st
JOIN trainers t ON st.studioid = t.studioid
GROUP BY st.name;
```
Tatsächliches Ergebnis: nur 5 Zeilen (5 bis 6 Trainer pro Studio). *MoveFit Wankdorf* hat noch keine Trainer und fehlt.

**Begründung der Änderungen:**

> [Hier eintragen]

In [ ]:
df_71 = abfrage("""
-- TODO: korrigierte Abfrage
""")
df_71

In [ ]:
# Selbstkontrolle 7.1
assert len(df_71) == 6 and dict(zip(df_71['name'], df_71['trainer_count'].astype(int)))['MoveFit Wankdorf'] == 0
print('7.1 richtig ✔')

### 7.2 Die Bedingung am falschen Ort
**Ziel:** Anzahl **besuchter** Buchungen (`status = 'attended'`) pro Studio — für **alle 6** Studios.
```sql
SELECT st.name, COUNT(b.bookingid) AS attended_count
FROM studios st
LEFT JOIN sessions s ON st.studioid = s.studioid
LEFT JOIN bookings b ON s.sessionid = b.sessionid
WHERE b.status = 'attended'
GROUP BY st.name;
```
Tatsächliches Ergebnis: nur 5 Zeilen — obwohl `LEFT JOIN` verwendet wird.

**Begründung der Änderungen:**

> [Hier eintragen]

In [ ]:
df_72 = abfrage("""
-- TODO: korrigierte Abfrage
""")
df_72

In [ ]:
# Selbstkontrolle 7.2
assert len(df_72) == 6 and int(df_72['attended_count'].sum()) == 1685
print('7.2 richtig ✔')

### 7.3 AND vor OR
**Ziel:** Anzahl **besuchter** Buchungen von Mitgliedern mit Mitgliedschaft `Student` **oder** `Senior`.
```sql
SELECT COUNT(*) AS n
FROM bookings b
JOIN members m ON b.memberid = m.memberid
WHERE m.membership_type = 'Student' OR m.membership_type = 'Senior' AND b.status = 'attended';
```
Das Ergebnis ist zu hoch.

**Begründung der Änderungen:**

> [Hier eintragen]

In [ ]:
df_73 = abfrage("""
-- TODO: korrigierte Abfrage
""")
df_73

### 7.4 Doppelt gezählt (Fan-out)
**Ziel:** Anzahl **verschiedener Mitglieder**, die im jeweiligen Studio mindestens einmal gebucht haben.
```sql
SELECT st.name, COUNT(b.memberid) AS member_count
FROM studios st
JOIN sessions s ON st.studioid = s.studioid
JOIN bookings b ON s.sessionid = b.sessionid
GROUP BY st.name
ORDER BY member_count DESC;
```
Tatsächliches Ergebnis: *MoveFit Oerlikon* 560 Mitglieder — das Studio hat aber deutlich weniger verschiedene Kunden.

**Begründung der Änderungen:**

> [Hier eintragen]

In [ ]:
df_74 = abfrage("""
-- TODO: korrigierte Abfrage
""")
df_74

In [ ]:
# Selbstkontrolle 7.4
assert dict(zip(df_74['name'], df_74['member_count'].astype(int)))['MoveFit Oerlikon'] == 335
print('7.4 richtig ✔')

### 7.5 Aggregat im WHERE
**Ziel:** Mitglieder (`memberid`) mit mehr als 8 Buchungen. Diese Abfrage bricht sogar mit einem Fehler ab:
```sql
SELECT memberid, COUNT(*) AS n
FROM bookings
WHERE COUNT(*) > 8
GROUP BY memberid;
```

**Begründung der Änderungen:**

> [Hier eintragen]

In [ ]:
df_75 = abfrage("""
-- TODO: korrigierte Abfrage
""")
df_75

---
## Stufe 8 — Konzeptfragen (wie Prüfungsteil A)

In der Prüfung zählen nur Antworten **mit Bezug zum Schema**. Nenne also konkrete Tabellen, Spalten und Beispiele.
Nutze für Belege ruhig eine kurze SQL-Abfrage.

### 8.1 Abgeleitete Daten speichern?
Ein Kollege möchte in `sessions` eine Spalte `booking_count` ergänzen, damit man die Anzahl Buchungen pro Termin nicht
jedes Mal zählen muss. Was spricht dafür, was dagegen?

**Ihre Antwort:**

> [Hier eintragen]

### 8.2 Data-Quality-Eigenschaften zuordnen
Ordne jedem Befund die passendste Eigenschaft zu (accurate, consistent, complete, reliable) und begründe kurz:

a) 15 Mitglieder haben keine E-Mail (`email` ist NULL).
b) `membership_type` enthält `Basic`, `basic`, `BASIC` und `Basik`.
c) Ein Mitglied hat das Geburtsjahr 1890.
d) Die Kursdaten stammen aus einer Excel-Liste, die jedes Studio selbst pflegt; niemand weiss, welche Version aktuell ist.

**Ihre Antwort:**

> [Hier eintragen]

### 8.3 Zwingend ein Fehler?
Der Termin `SE0005` hat eine Kapazität von 12 Plätzen, aber 13 nicht stornierte Buchungen. Ist das **zwingend** ein
Datenfehler? Begründe in einem Satz.

**Ihre Antwort:**

> [Hier eintragen]

### 8.4 Warum zwei Preise?
Warum enthält `bookings` eine eigene Spalte `paid_price`, obwohl der Preis schon in `sessions.base_price` steht?
Belege deine Antwort mit einer Zahl aus der Datenbank (Abfrage in der Code-Zelle).

In [ ]:
df_84 = abfrage("""
-- TODO: Wie viele Buchungen haben einen paid_price unter dem base_price?
""")
df_84

**Ihre Antwort:**

> [Hier eintragen]

In [ ]:
# Selbstkontrolle 8.4
assert int(df_84['n'][0]) == 1516
print('8.4 richtig ✔')

---
**Geschafft!** Weiter mit der Probeprüfung `11_probepruefung_ap01.ipynb` — setze dir dafür ein Zeitlimit wie in der
echten Prüfung und arbeite ohne Lösung.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')